# 06b — Full Supervised Baseline — ELECTRA-small

Same as `06_full_supervised_baseline.ipynb`, run with
`utils.config.CLASSIFIER_MODEL_NAME_ALT` (`google/electra-small-discriminator`)
instead of DistilBERT, at the same `CLASSIFIER_SAMPLE_SIZE`, for a direct
side-by-side comparison in `07_comparison.ipynb`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_semisupervised
from utils.modeling import get_predictions, train_model

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

train_sample = stratified_sample(train_clean, config.CLASSIFIER_SAMPLE_SIZE, seed=config.SEED)
print(f"Training on {len(train_sample)} fully-labeled rows (ELECTRA-small upper bound baseline)")

Training on 152 fully-labeled rows (ELECTRA-small upper bound baseline)


In [3]:
model, tokenizer = train_model(train_sample, model_name=config.CLASSIFIER_MODEL_NAME_ALT, epochs=3)

test_probs = get_predictions(model, tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_full_supervised_electra.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_full_supervised_electra.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved full-supervised baseline (ELECTRA-small) results.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstre

C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


              precision    recall  f1-score   support

       World       0.60      0.50      0.55      1900
      Sports       0.34      0.91      0.49      1900
    Business       0.50      0.03      0.06      1900
    Sci/Tech       0.49      0.21      0.29      1900

    accuracy                           0.41      7600
   macro avg       0.48      0.41      0.35      7600
weighted avg       0.48      0.41      0.35      7600

Saved full-supervised baseline (ELECTRA-small) results.


In [4]:
from utils.samples import save_label_samples

save_label_samples(
    test_clean["text"], test_preds, test_clean["label"].to_numpy(),
    config.CLASS_NAMES, confidence=test_probs.max(axis=1), n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_full_supervised_electra.csv")
print("Saved sample generated labels for full_supervised_electra.")

Saved sample generated labels for full_supervised_electra.
